# Powerflow Headline and Duration Summary

Compact Step 4 comparison view for DB-backed `--summary-only` power-flow runs. The headline violin uses stored critical-tail hourly values: p99 upper-tail loading for transformers/cables and p01 lower-tail voltage for load buses. The duration plot uses stored annual time-percentiles, without loading full timestep power-flow tables.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
root = next((p for p in [cwd, *cwd.parents] if (p / "GridExpand").exists()), None)
if root is None:
    raise RuntimeError("Could not resolve repository root containing GridExpand/.")

gridexpand_dir = root / "GridExpand"
plotting_dir = gridexpand_dir / "5.postprocessing" / "plotting"
for path in (gridexpand_dir, plotting_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

root


In [ ]:
from IPython.display import display
import importlib
import pandas as pd

import powerflow_plotting
importlib.reload(powerflow_plotting)

from powerflow_plotting import (
    plot_powerflow_headline_asset_violins,
    plot_powerflow_percentile_profiles,
    powerflow_tail_duration_data_db,
    tail_threshold_counts,
)

PLZ = 91301
AGS = None
RUN_NAME = "baseline_synthetic_hh_only"
STAGE = "pre"

LOADING_THRESHOLD_PERCENT = 100.0
VOLTAGE_THRESHOLD_PU = 0.90


In [ ]:
grid_summary, tail_values, percentile_profile = powerflow_tail_duration_data_db(
    run_name=RUN_NAME,
    stage=STAGE,
    ags=AGS,
    plz=PLZ,
)
threshold_counts = tail_threshold_counts(
    tail_values,
    loading_threshold_percent=LOADING_THRESHOLD_PERCENT,
    voltage_threshold_pu=VOLTAGE_THRESHOLD_PU,
)

print(
    f"Loaded {len(grid_summary)} grid row(s), "
    f"{len(tail_values)} critical-tail value row(s), "
    f"and {len(percentile_profile)} percentile profile row(s)."
)


In [ ]:
# display(
#     grid_summary[[
#         "grid",
#         "powerflow_run_id",
#         "n_timesteps",
#         "n_voltage_buses",
#         "n_cables",
#         "trafo_loading_p99_time_percent",
#         "trafo_loading_max_time_percent",
#         "cable_loading_p95_asset_percent",
#         "voltage_p05_load_bus_hour_pu",
#     ]].style.format({
#         "trafo_loading_p99_time_percent": "{:.2f}",
#         "trafo_loading_max_time_percent": "{:.2f}",
#         "cable_loading_p95_asset_percent": "{:.2f}",
#         "voltage_p05_load_bus_hour_pu": "{:.4f}",
#     })
# )

# display(
#     tail_values.groupby(["metric", "asset_type", "tail"])["value"]
#     .describe()
#     .style.format("{:.4f}")
# )

# display(
#     threshold_counts.groupby(["metric", "asset_type", "is_complete_for_threshold"])[
#         "n_tail_hours_beyond_threshold"
#     ]
#     .agg(["count", "sum", "max", "mean"])
#     .style.format("{:.2f}")
# )


In [ ]:
display(plot_powerflow_headline_asset_violins(tail_values, show=False))

In [ ]:
display(plot_powerflow_percentile_profiles(percentile_profile, show=False))

## Metric Notes

- Transformer/cable violins show all stored hourly loading values at or above each asset's own p99 loading threshold.
- Voltage violins show all stored hourly voltage values at or below each load bus's own p01 voltage threshold.
- Percentile profiles summarize the annual duration shape: loading uses `p50`, `p90`, `p95`, `p99`, `max`; voltage uses `p50`, `p10`, `p05`, `p01`, `min`.
- Threshold counts from `tail_threshold_counts(...)` are exact only when the chosen threshold is at least as extreme as the stored tail threshold. Otherwise they are lower bounds, because non-tail hours were not stored.
- The line is the median across assets at each time-percentile. Example: at cable p95, it is the median of all cables’ p95 loading values.
- The shadow is the asset spread: p10 to p90 across assets at that same time-percentile. Example: at cable p95, the shaded range covers the 10th to 90th percentile of cable p95 loading values across all cables.
